import pandas as pd
print(pd.__version__)

In [1]:
import pandas as pd
print(pd.__version__)

3.1.1


In [2]:
df = pd.read_csv("trips_lyon.csv")
print(df.shape)

(7000, 12)


In [3]:
import numpy as np
import pandas as pd

# 读取原始数据，并在副本中标准化空白。
raw = pd.read_csv("trips_lyon.csv", dtype="string", keep_default_na=False)
trips = raw.copy()
for col in trips.columns:
    trips[col] = trips[col].str.strip().replace("", pd.NA)

# 记录问题及受影响行数。各项可能涉及同一行。
audit = []

def report(column, issue, mask):
    audit.append((column, issue, int(mask.fillna(False).sum())))

for col in trips.columns:
    report(col, "缺失", trips[col].isna())

duplicate = trips.duplicated()
report("整行", "重复记录（首条之后）", duplicate)

identified = trips["trip_id"].notna() & ~duplicate
conflicts = trips.loc[identified, "trip_id"].duplicated(keep=False)
report("trip_id", "去除整行重复后，同一 ID 仍对应多条记录", conflicts)

# 校验时间和站点格式，保留无法确认的站点代码供复核。
time_text = trips["start_time"]
time_format = time_text.str.fullmatch(
    r"[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}",
    na=False,
)
trips["start_time"] = pd.to_datetime(
    time_text.where(time_format), format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
report("start_time", "格式异常或无效日期", time_text.notna() & trips["start_time"].isna())

for col in ["start_station_id", "end_station_id"]:
    bad_code = trips[col].notna() & ~trips[col].str.fullmatch(r"ST-[0-9]+", na=False)
    report(col, "站点代码格式异常", bad_code)
    trips[f"{col}_format_flag"] = bad_code

# 将无效数值置空，并单独统计各项检查结果。
numeric_cols = [
    "duration_minutes", "distance_km", "distance_meters",
    "battery_start_pct", "price_eur", "applied_rate_per_min",
]
originally_missing = trips[numeric_cols].isna()
for col in numeric_cols:
    parsed = pd.to_numeric(trips[col], errors="coerce")
    finite = parsed.notna() & ~parsed.isin([np.inf, -np.inf])
    report(col, "非数值或无穷值", trips[col].notna() & ~finite)

    invalid = parsed.lt(0)
    if col == "duration_minutes":
        invalid = parsed.le(0)
    elif col == "battery_start_pct":
        invalid = parsed.lt(0) | parsed.gt(100)

    out_of_range = finite & invalid.fillna(False)
    report(col, "超出有效范围", out_of_range)
    trips[col] = parsed.where(finite & ~out_of_range)

# 检查双单位距离。示例允许 1 米的舍入差。
km = trips["distance_km"]
meters = trips["distance_meters"]
mismatch = ((km * 1000 - meters).abs() > 1).fillna(False)
report("distance_km / distance_meters", "换算差超过 1 米", mismatch)
trips["distance_mismatch_flag"] = mismatch

# 按顺序删除重复行及缺少行程 ID 的行，并报告每次实际删除数。
clean = trips.loc[~duplicate].copy()
missing_id = clean["trip_id"].isna()
removed_missing_id = int(missing_id.sum())
clean = clean.loc[~missing_id].copy()
clean["trip_id_conflict_flag"] = clean["trip_id"].duplicated(keep=False)

# 从有效的另一列补齐原本缺失的距离。
fill_km = originally_missing.loc[clean.index, "distance_km"]
fill_meters = originally_missing.loc[clean.index, "distance_meters"]
clean.loc[fill_km, "distance_km"] = clean.loc[fill_km, "distance_meters"] / 1000
clean.loc[fill_meters, "distance_meters"] = clean.loc[fill_meters, "distance_km"] * 1000

# 用 1.5 倍 IQR 标记异常。距离阈值排除两列距离不一致的记录。
for col in ["duration_minutes", "distance_km"]:
    eligible = clean[col].notna()
    if col == "distance_km":
        eligible &= ~clean["distance_mismatch_flag"]

    sample = clean.loc[eligible, col]
    flag = pd.Series(False, index=clean.index)
    if len(sample) >= 4:
        q1, q3 = sample.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr > 0:
            flag.loc[sample.index] = (
                sample.lt(q1 - 1.5 * iqr) | sample.gt(q3 + 1.5 * iqr)
            )
    clean[f"{col}_outlier_flag"] = flag
    report(col, "IQR 异常（基于保留行）", flag)

# 输出审计表、行数核对和清洗文件。
print(pd.DataFrame(audit, columns=["字段", "问题", "受影响行数"]).to_string(index=False))
print(f"原始行数 = {len(raw)}")
print(f"删除重复行 = {int(duplicate.sum())}")
print(f"删除缺少 trip_id 的行（去重后） = {removed_missing_id}")
print(f"最终行数 = {len(clean)}")
assert len(clean) == len(raw) - int(duplicate.sum()) - removed_missing_id
clean.to_csv("trips_lyon_clean.csv", index=False)

                           字段                    问题  受影响行数
                      trip_id                    缺失      0
                      user_id                    缺失      0
                   start_time                    缺失      0
             start_station_id                    缺失      0
               end_station_id                    缺失    407
                 vehicle_type                    缺失      0
             duration_minutes                    缺失      0
                  distance_km                    缺失      0
              distance_meters                    缺失      0
            battery_start_pct                    缺失      0
                    price_eur                    缺失      0
         applied_rate_per_min                    缺失      0
                           整行            重复记录（首条之后）      0
                      trip_id 去除整行重复后，同一 ID 仍对应多条记录      0
                   start_time             格式异常或无效日期      0
             start_station_id              站点代码格式异常     

In [5]:
import pandas as pd
import numpy as np

# 1. Load raw data and inspect total row and column counts without modifying the original DataFrame
df_raw = pd.read_csv("trips_lyon.csv")
print(f"Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

# 2. Audit all columns before cleaning and record data quality issues
audit_issues = []
audit_issues.append(("trip_id", "Missing or duplicate trip_id", df_raw["trip_id"].isna().sum() + df_raw["trip_id"].duplicated().sum()))
audit_issues.append(("user_id", "Missing user_id", df_raw["user_id"].isna().sum()))
parsed_time = pd.to_datetime(df_raw["start_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
audit_issues.append(("start_time", "Missing or invalid timestamp format", parsed_time.isna().sum()))
valid_start_st = df_raw["start_station_id"].astype(str).str.match(r"^ST-\d+$", na=False)
valid_end_st = df_raw["end_station_id"].astype(str).str.match(r"^ST-\d+$", na=False)
audit_issues.append(("start_station_id", "Missing or invalid station format (expected ST-XXX)", (~valid_start_st).sum()))
audit_issues.append(("end_station_id", "Missing or invalid station format (expected ST-XXX)", (~valid_end_st).sum()))
audit_issues.append(("vehicle_type", "Missing or unstandardized casing/whitespace", df_raw["vehicle_type"].isna().sum() + (df_raw["vehicle_type"].dropna() != df_raw["vehicle_type"].dropna().str.strip().str.lower()).sum()))
audit_issues.append(("duration_minutes", "Missing, non-positive (<= 0), or extreme outliers (> 1440 min)", (df_raw["duration_minutes"].isna() | (df_raw["duration_minutes"] <= 0) | (df_raw["duration_minutes"] > 1440)).sum()))
audit_issues.append(("distance_km / distance_meters", "Missing, negative (< 0), or unit mismatch (> 0.05 km diff)", (df_raw["distance_km"].isna() | df_raw["distance_meters"].isna() | (df_raw["distance_km"] < 0) | (df_raw["distance_meters"] < 0) | ((df_raw["distance_km"] - df_raw["distance_meters"] / 1000.0).abs() > 0.05)).sum()))
audit_issues.append(("battery_start_pct", "Missing or out of valid range [0, 100]", (df_raw["battery_start_pct"].isna() | (df_raw["battery_start_pct"] < 0) | (df_raw["battery_start_pct"] > 100)).sum()))
audit_issues.append(("price_eur / applied_rate_per_min", "Missing or negative pricing values (< 0)", (df_raw["price_eur"].isna() | df_raw["applied_rate_per_min"].isna() | (df_raw["price_eur"] < 0) | (df_raw["applied_rate_per_min"] < 0)).sum()))

print("\n--- (1) Problems Found (Pre-Cleaning Audit) ---")
for idx, (col, desc, count) in enumerate(audit_issues, 1):
    print(f"{idx}. Column: {col} | Issue: {desc} | Rows affected: {count}")

# 3. Create a working copy so the raw DataFrame remains untouched
df_clean = df_raw.copy()
removal_log = []

# 4. Report and drop rows with missing critical identifiers or duplicate trip_id
before = len(df_clean)
missing_ids = df_clean["trip_id"].isna() | df_clean["user_id"].isna()
print(f"\n[Cleaning Step] Missing trip_id or user_id count: {missing_ids.sum()}")
df_clean = df_clean[~missing_ids].drop_duplicates(subset=["trip_id"], keep="first")
removed = before - len(df_clean)
removal_log.append(("Missing trip_id/user_id or duplicate trip_id", removed))
print(f"Removed {removed} rows -> Reason: Primary/foreign keys cannot be null or duplicated. Remaining: {len(df_clean)}")

# 5. Parse start_time to datetime and remove rows with unparseable timestamps
before = len(df_clean)
df_clean["start_time"] = pd.to_datetime(df_clean["start_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
invalid_time = df_clean["start_time"].isna()
print(f"[Cleaning Step] Invalid/missing start_time count: {invalid_time.sum()}")
df_clean = df_clean[~invalid_time]
removed = before - len(df_clean)
removal_log.append(("Missing or invalid start_time format", removed))
print(f"Removed {removed} rows -> Reason: Invalid timestamp format (expected YYYY-MM-DD HH:MM:SS). Remaining: {len(df_clean)}")

# 6. Validate station ID format (ST-XXX) and drop rows with missing or malformed station codes
before = len(df_clean)
df_clean["start_station_id"] = df_clean["start_station_id"].astype(str).str.strip().str.upper()
df_clean["end_station_id"] = df_clean["end_station_id"].astype(str).str.strip().str.upper()
valid_stations = df_clean["start_station_id"].str.match(r"^ST-\d+$") & df_clean["end_station_id"].str.match(r"^ST-\d+$")
df_clean = df_clean[valid_stations]
removed = before - len(df_clean)
removal_log.append(("Invalid or missing start_station_id / end_station_id", removed))
print(f"Removed {removed} rows -> Reason: Station IDs must match format ST-XXX to link with stations_metadata.csv. Remaining: {len(df_clean)}")

# 7. Standardize vehicle_type text and drop rows where vehicle_type is missing
before = len(df_clean)
missing_vehicle = df_clean["vehicle_type"].isna()
print(f"[Cleaning Step] Missing vehicle_type count: {missing_vehicle.sum()}")
df_clean = df_clean[~missing_vehicle]
df_clean["vehicle_type"] = df_clean["vehicle_type"].astype(str).str.strip().str.lower()
removed = before - len(df_clean)
removal_log.append(("Missing vehicle_type", removed))
print(f"Removed {removed} rows -> Reason: Vehicle type is required to analyze bike vs. scooter usage and pricing. Remaining: {len(df_clean)}")

# 8. Reconcile distance_km and distance_meters, drop negative values, and reconcile unit mismatches
before = len(df_clean)
both_dist_missing = df_clean["distance_km"].isna() & df_clean["distance_meters"].isna()
print(f"[Cleaning Step] Both distance_km and distance_meters missing: {both_dist_missing.sum()}")
df_clean = df_clean[~both_dist_missing]
df_clean["distance_km"] = df_clean["distance_km"].fillna(df_clean["distance_meters"] / 1000.0)
df_clean["distance_meters"] = df_clean["distance_meters"].fillna(df_clean["distance_km"] * 1000.0)
invalid_dist = (df_clean["distance_km"] < 0) | (df_clean["distance_meters"] < 0) | ((df_clean["distance_km"] - df_clean["distance_meters"] / 1000.0).abs() > 0.05)
df_clean = df_clean[~invalid_dist]
removed = before - len(df_clean)
removal_log.append(("Missing, negative, or conflicting distance_km & distance_meters", removed))
print(f"Removed {removed} rows -> Reason: Negative distances or irreconcilable km/meter discrepancies (> 50m). Remaining: {len(df_clean)}")

# 9. Remove missing or out-of-range battery_start_pct values outside [0, 100]
before = len(df_clean)
invalid_battery = df_clean["battery_start_pct"].isna() | (df_clean["battery_start_pct"] < 0) | (df_clean["battery_start_pct"] > 100)
print(f"[Cleaning Step] Missing or out-of-range battery_start_pct count: {invalid_battery.sum()}")
df_clean = df_clean[~invalid_battery]
removed = before - len(df_clean)
removal_log.append(("Missing or invalid battery_start_pct (<0 or >100)", removed))
print(f"Removed {removed} rows -> Reason: Battery percentage must be within [0, 100]. Remaining: {len(df_clean)}")

# 10. Remove missing, non-positive, or extreme duration_minutes and negative pricing records
before = len(df_clean)
invalid_duration_price = (
    df_clean["duration_minutes"].isna() | (df_clean["duration_minutes"] <= 0) |
    df_clean["price_eur"].isna() | (df_clean["price_eur"] < 0) |
    df_clean["applied_rate_per_min"].isna() | (df_clean["applied_rate_per_min"] < 0)
)
print(f"[Cleaning Step] Invalid duration_minutes (<=0/NaN) or negative/NaN pricing count: {invalid_duration_price.sum()}")
df_clean = df_clean[~invalid_duration_price]
removed = before - len(df_clean)
removal_log.append(("Non-positive duration_minutes or negative/missing price/rate", removed))
print(f"Removed {removed} rows -> Reason: Duration must be > 0 and pricing fields must be non-negative and non-null. Remaining: {len(df_clean)}")

# 11. Detect and remove statistical outliers in duration_minutes, distance_km, and price_eur using IQR per vehicle_type
before = len(df_clean)
outlier_mask = pd.Series(False, index=df_clean.index)
for col in ["duration_minutes", "distance_km", "price_eur"]:
    q1 = df_clean.groupby("vehicle_type")[col].transform(lambda s: s.quantile(0.25))
    q3 = df_clean.groupby("vehicle_type")[col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    lower_bound = q1 - 3.0 * iqr
    upper_bound = q3 + 3.0 * iqr
    outlier_mask = outlier_mask | (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)

df_clean = df_clean[~outlier_mask]
removed = before - len(df_clean)
removal_log.append(("Extreme outliers (3*IQR by vehicle_type) in duration, distance, or price", removed))
print(f"Removed {removed} rows -> Reason: Extreme statistical outliers distort pricing model calibration. Remaining: {len(df_clean)}")

# 12. Export cleaned dataset and print final results summary
df_clean.to_csv("trips_lyon_clean.csv", index=False)

print("\n--- (3) Results Summary ---")
print(f"Raw rows: {len(df_raw)}")
for rule, count in removal_log:
    print(f" - Removed ({rule}): {count}")
print(f"Total rows removed: {len(df_raw) - len(df_clean)}")
print(f"Final row count: {len(df_clean)}")

Raw dataset shape: 7000 rows, 12 columns

--- (1) Problems Found (Pre-Cleaning Audit) ---
1. Column: trip_id | Issue: Missing or duplicate trip_id | Rows affected: 0
2. Column: user_id | Issue: Missing user_id | Rows affected: 0
3. Column: start_time | Issue: Missing or invalid timestamp format | Rows affected: 0
4. Column: start_station_id | Issue: Missing or invalid station format (expected ST-XXX) | Rows affected: 0
5. Column: end_station_id | Issue: Missing or invalid station format (expected ST-XXX) | Rows affected: 407
6. Column: vehicle_type | Issue: Missing or unstandardized casing/whitespace | Rows affected: 2257
7. Column: duration_minutes | Issue: Missing, non-positive (<= 0), or extreme outliers (> 1440 min) | Rows affected: 205
8. Column: distance_km / distance_meters | Issue: Missing, negative (< 0), or unit mismatch (> 0.05 km diff) | Rows affected: 0
9. Column: battery_start_pct | Issue: Missing or out of valid range [0, 100] | Rows affected: 0
10. Column: price_eur / a